# Day 90 (halfway!): `cleaning_report` as a function

**Goal:** yesterday I wrote the report by hand, row by row. Today I make a function that prints the same report for any table with one line.

**The picture:** a rubber stamp. Design the report card once, then stamp it on any class.

> Restart the kernel, then run top to bottom.

## 0. Setup

In [ ]:
import sys
sys.path.append("../..")

import pandas as pd
from agronomic_functions import build_flat, quality_flags, dataset_alarm, cleaning_report
from db_config import run_query

## 1. The lesson: three small ideas
1. **A loop builds one row for each column.** `rows.append({...})` drops one new row into a list. After the loop, `pd.DataFrame(rows)` makes the table.
2. **The action can be decided by code.** `if`, `elif` and `else`: Python reads from the top and stops at the first line that is true.
3. **A function returns the table.** It builds the rows inside and ends with `return pd.DataFrame(rows)`. It does not print it (the vending machine from Day 88).

## 2. Exercise on made-up data
**Prediction before running:** `a` has 1 missing (25%), `b` has 3 (75%), `c` has none (0%). Actions: `keep`, `keep; 75% missing...`, `none needed`.

In [ ]:
toy = pd.DataFrame({
    "a": [1, 2, None, 4],
    "b": [None, None, None, 4],
    "c": [1, 2, 3, 4]
})

rows = []

for col in ["a", "b", "c"]:
    missing = int(toy[col].isna().sum())          # 1. count the missing values
    pct = missing / len(toy) * 100                # 2. the percentage

    if pct == 0:                                  # 3. the traffic-light rule
        action = "none needed"
    elif pct <= 25:
        action = "keep"
    else:
        action = f"keep; {pct:.0f}% missing, usable only for the rest"

    rows.append({"check": f"Missing {col}", "result": missing, "action": action})   # 4. drop the row in the bucket

report = pd.DataFrame(rows)                       # 5. turn the bucket into a table
print(report)

**Check question: why a loop and not three copy-pasted dictionaries?**
1. Less typing and fewer typos: with copy-paste I might forget to change a column name in one row, a silent bug.
2. It scales: for 50 columns I just add names to the list.

## 3. The function
`cleaning_report(df, columns, key)` lives in `csa_soil_project/agronomic_functions.py`. It:
1. **guards** (a missing column, a missing key, an empty table, which would divide by zero)
2. writes the fixed rows: `Total layers`, `Duplicate rows`, `Duplicate <key>`
3. loops over `columns` for one `Missing <col>` row each, with the traffic-light rule
4. **returns** the table

**The bug I fixed after review:** the two duplicate rows had `"action": "none needed"` typed in, so a real duplicate would still say "none needed". The action must **follow the result**: count 0 gives `none needed`; above 0 gives `investigate; ...`.

In [ ]:
help(cleaning_report)

## 4. The test (on made-up data)
Expected: 4 layers, 0 duplicate rows, 1 duplicate id, `a` 1 missing, `b` 3 missing (75%).

In [ ]:
def test_cleaning_report():
    toy = pd.DataFrame({
        "id": [1, 2, 2, 4],
        "a":  [1, None, 3, 4],
        "b":  [None, None, None, 4]
    })

    report = cleaning_report(toy, ["a", "b"], key="id")

    # Look up a report cell by its check name
    res = report.set_index("check")["result"]
    act = report.set_index("check")["action"]

    assert res["Total layers"] == 4
    assert res["Duplicate rows"] == 0
    assert res["Duplicate id"] == 1
    assert res["Missing a"] == 1
    assert res["Missing b"] == 3

    assert "75%" in act["Missing b"], "Action for b should contain 75%"
    assert act["Duplicate id"] != "none needed", "A duplicate must not say none needed"

    # A missing column raises ValueError
    try:
        cleaning_report(toy, ["wrong_col"], key="id")
        raise AssertionError("Did not raise ValueError for missing column")
    except ValueError:
        pass

    # An empty table raises ValueError
    try:
        cleaning_report(toy.iloc[0:0], ["a", "b"], key="id")
        raise AssertionError("Did not raise ValueError for empty table")
    except ValueError:
        pass

    print("all tests passed")

test_cleaning_report()

## 5. The real table
**Prediction:** it reproduces Day 89: 1,756 layers, 0, 0, then 14, 1,257 and 1,256 missing.

In [ ]:
flat = build_flat("orgc", ["ph", "clay", "sand"])
checked = quality_flags(flat)
meta = run_query("SELECT layer_id, dataset_id FROM wosis_orgc WHERE country_name = 'Ghana'")
traced = pd.merge(checked, meta, on="layer_id", how="left")
print(len(traced))        # must be 1756

In [ ]:
cleaning_report(traced, ["ph", "clay", "sand"], key="layer_id")

**Why `key` is a parameter, not typed inside:** the function should work for any table. For a rainfall table, the unique identifier might be `station_id`: `cleaning_report(rain_df, ["rainfall"], key="station_id")`.

## 6. What I keep from today
- A loop builds one row per column.
- Look up report cells with `set_index("check")`.
- **The `action` must follow the `result`.** A typed "none needed" would hide a real duplicate. A test that checks only numbers misses it, so test the action text too.
- A print-based "FAIL" cannot stop a test. Use `raise AssertionError`.

**Next:** Day 91, a script that runs the whole check with one command.